# Batch lick-bout analysis from `.doric` files alone

Use this workflow when vendor lifetime, licking, and Ensure TTLs are embedded in each FluoPulse file. No behavior or tau CSV export is required, and no NI-DAQ file is needed unless external behavior channels must be added. Descriptive names such as `SC43_250625_15secFoodReward_40trials.doric` are retained as recording labels.

In [ ]:
from datetime import date, datetime
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import numpy as np
from openpyxl import load_workbook
from fluopulse_analysis import extract_perievent, find_doric_files, find_lick_bouts, read_doric

## Configuration

The slow processing stage always aligns every qualifying bout and saves it to `all_bout_cache.npz`. After that, use the fast filter cell to choose Ensure status, mice, dates, or recording-description text without rereading the `.doric` files. An Ensure-associated bout has an embedded Ensure onset from 5 seconds before through 1 second after its first lick.

In [ ]:
SESSION_TABLE = Path.home() / "Documents" / "SC_FLiPmice_2025.xlsx"
DORIC_ROOT = Path(r"Z:\FLIM FLIP")
OUTPUT_FOLDER = Path.cwd() / "analysis_output_doric_only_lick_bouts"
CACHE_FILE = OUTPUT_FOLDER / "all_bout_cache.npz"
MAX_INTERLICK_GAP_S = 1.0
MIN_LICKS_PER_BOUT = 5
ENSURE_WINDOW_S = (-5.0, 1.0)
PERIEVENT_WINDOW_S = (-10.0, 10.0)
BASELINE_WINDOW_S = (-10.0, 0.0)
SAMPLE_INTERVAL_S = 0.1
print(f"Doric root available: {DORIC_ROOT.is_dir()} ({DORIC_ROOT})")

In [ ]:
def normalize_date(value):
    if isinstance(value, (date, datetime)):
        return value.strftime("%y%m%d")
    return str(int(value)).zfill(6)

workbook = load_workbook(SESSION_TABLE, data_only=True, read_only=True)
worksheet = workbook.active
rows = list(worksheet.iter_rows(values_only=True))
workbook.close()
headers = [str(value).strip() if value is not None else "" for value in rows[0]]
if not headers or headers[0].casefold() != "mouse":
    raise ValueError("The first workbook column must be named 'mouse'")
session_records = []
for row in rows[1:]:
    if not row or row[0] is None:
        continue
    mouse = str(row[0]).strip()
    for column, value in zip(headers[1:], row[1:], strict=False):
        if value is not None:
            session_records.append({"mouse": mouse, "date": normalize_date(value), "source_column": column})
print(f"Loaded {len(session_records)} mouse-date sessions")

In [ ]:
def recording_label(path, mouse, session_date):
    stem = path.stem
    for prefix in (f"{mouse}_{session_date}_", f"{session_date}_{mouse}_"):
        if stem.casefold().startswith(prefix.casefold()):
            return stem[len(prefix):]
    return stem

from time import perf_counter

qc_rows = []
bout_rows = []
aligned_trials = []
aligned_lick_counts = []
trial_mice = []
trial_has_ensure = []
relative_time = None
processing_started = perf_counter()
total_sessions = len(session_records)
print(f"Starting {total_sessions} mouse-date sessions", flush=True)
for session_index, record in enumerate(session_records, start=1):
    mouse = record["mouse"]
    session_date = record["date"]
    print(f"[{session_index}/{total_sessions}] {mouse} {session_date}: searching for recordings...", flush=True)
    doric_files = find_doric_files(mouse, session_date, doric_root=DORIC_ROOT)
    if not doric_files:
        qc_rows.append({"mouse": mouse, "date": session_date, "recording": "", "status": f"missing .doric under {DORIC_ROOT / mouse}"})
        print("    no matching .doric files", flush=True)
        continue
    print(f"    found {len(doric_files)} recording(s)", flush=True)
    for file_index, doric_file in enumerate(doric_files, start=1):
        label = recording_label(doric_file, mouse, session_date)
        file_started = perf_counter()
        print(f"    ({file_index}/{len(doric_files)}) {label}: reading...", flush=True)
        try:
            recording = read_doric(doric_file)
            lick_times = recording.digital_pulses.get("licking")
            ensure_pulses = recording.digital_pulses.get("ensure")
            if lick_times is None:
                raise ValueError("No embedded lick channel")
            ensure_times = np.array([]) if ensure_pulses is None else ensure_pulses.onset_times
            bouts = find_lick_bouts(
                lick_times.onset_times,
                max_interlick_gap_seconds=MAX_INTERLICK_GAP_S,
                min_licks=MIN_LICKS_PER_BOUT,
            )
            lick_onsets = lick_times.onset_times
            previous_indices = np.searchsorted(lick_onsets, bouts.onset_times, side="left") - 1
            previous_reference = np.full(bouts.onset_times.shape, recording.time[0], dtype=float)
            has_previous_lick = previous_indices >= 0
            previous_reference[has_previous_lick] = lick_onsets[previous_indices[has_previous_lick]]
            preceding_lick_gap_s = bouts.onset_times - previous_reference
            has_ensure = np.array([
                np.any(
                    (ensure_times >= onset + ENSURE_WINDOW_S[0])
                    & (ensure_times <= onset + ENSURE_WINDOW_S[1])
                )
                for onset in bouts.onset_times
            ], dtype=bool)
            selected = np.ones(bouts.onset_times.size, dtype=bool)
            qc_rows.append({
                "mouse": mouse,
                "date": session_date,
                "recording": label,
                "status": "ok",
                "duration_s": recording.time[-1] - recording.time[0],
                "lifetime_rate_hz": recording.sampling_frequency,
                "median_tau_ns": np.nanmedian(recording.tau_ns),
                "median_fit_r_square": np.nanmedian(recording.r_square),
                "lick_count": lick_times.onset_times.size,
                "bout_count": bouts.onset_times.size,
                "selected_bout_count": int(selected.sum()),
                "ensure_count": ensure_times.size,
            })
            print(f"        {lick_times.onset_times.size} licks, {bouts.onset_times.size} bouts, {int(selected.sum())} selected", flush=True)
            if not np.any(selected):
                print(f"        complete in {perf_counter() - file_started:.1f} s", flush=True)
                continue
            selected_indices = np.flatnonzero(selected)
            relative_time, trials = extract_perievent(
                recording.time,
                recording.tau_ns,
                bouts.onset_times[selected],
                window_seconds=PERIEVENT_WINDOW_S,
                sample_interval_seconds=SAMPLE_INTERVAL_S,
            )
            relative_time[np.isclose(relative_time, 0.0, atol=SAMPLE_INTERVAL_S / 1000)] = 0.0
            baseline_mask = (relative_time >= BASELINE_WINDOW_S[0]) & (relative_time < BASELINE_WINDOW_S[1])
            baseline = np.nanmean(trials[:, baseline_mask], axis=1, keepdims=True)
            trials = trials - baseline
            lick_bin_edges = np.concatenate([
                relative_time - SAMPLE_INTERVAL_S / 2,
                [relative_time[-1] + SAMPLE_INTERVAL_S / 2],
            ])
            lick_count_trials = []
            for index in selected_indices:
                relative_licks = lick_onsets - bouts.onset_times[index]
                relative_licks = relative_licks[
                    (relative_licks >= relative_time[0]) & (relative_licks <= relative_time[-1])
                ]
                lick_count_trials.append(np.histogram(relative_licks, bins=lick_bin_edges)[0])
            lick_count_trials = np.vstack(lick_count_trials)
            for local_index, bout_index in enumerate(selected_indices):
                ensure_candidates = ensure_times[
                    (ensure_times >= bouts.onset_times[bout_index] + ENSURE_WINDOW_S[0])
                    & (ensure_times <= bouts.onset_times[bout_index] + ENSURE_WINDOW_S[1])
                ]
                ensure_latency = float(ensure_candidates[-1] - bouts.onset_times[bout_index]) if ensure_candidates.size else np.nan
                bout_rows.append({
                    "mouse": mouse,
                    "date": session_date,
                    "recording": label,
                    "source_file": str(doric_file),
                    "bout_index": int(bout_index),
                    "bout_onset_s": bouts.onset_times[bout_index],
                    "bout_end_s": bouts.offset_times[bout_index],
                    "lick_count": int(bouts.lick_counts[bout_index]),
                    "preceding_lick_gap_s": preceding_lick_gap_s[bout_index],
                    "has_ensure": bool(has_ensure[bout_index]),
                    "ensure_latency_s": ensure_latency,
                })
                aligned_trials.append(trials[local_index])
                aligned_lick_counts.append(lick_count_trials[local_index])
                trial_mice.append(mouse)
                trial_has_ensure.append(bool(has_ensure[bout_index]))
            print(f"        complete in {perf_counter() - file_started:.1f} s", flush=True)
        except Exception as error:
            qc_rows.append({"mouse": mouse, "date": session_date, "recording": label, "status": f"error: {error}"})
            print(f"        ERROR after {perf_counter() - file_started:.1f} s: {error}", flush=True)
print(f"Finished in {perf_counter() - processing_started:.1f} s", flush=True)
print(f"Checked {len(qc_rows)} recording records", flush=True)
print(f"Aligned {len(aligned_trials)} total lick bouts", flush=True)
if not aligned_trials:
    raise ValueError("No lick bouts passed the processing criteria")
all_trials = np.vstack(aligned_trials)
all_lick_count_trials = np.vstack(aligned_lick_counts)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    CACHE_FILE,
    relative_time_s=relative_time,
    delta_tau_ns=all_trials,
    lick_counts_per_bin=all_lick_count_trials,
    lick_bin_width_s=np.asarray(SAMPLE_INTERVAL_S),
    mouse=np.asarray([row["mouse"] for row in bout_rows], dtype=str),
    session_date=np.asarray([row["date"] for row in bout_rows], dtype=str),
    recording=np.asarray([row["recording"] for row in bout_rows], dtype=str),
    source_file=np.asarray([row["source_file"] for row in bout_rows], dtype=str),
    bout_index=np.asarray([row["bout_index"] for row in bout_rows], dtype=int),
    bout_onset_s=np.asarray([row["bout_onset_s"] for row in bout_rows], dtype=float),
    bout_end_s=np.asarray([row["bout_end_s"] for row in bout_rows], dtype=float),
    lick_count=np.asarray([row["lick_count"] for row in bout_rows], dtype=int),
    preceding_lick_gap_s=np.asarray([row["preceding_lick_gap_s"] for row in bout_rows], dtype=float),
    has_ensure=np.asarray([row["has_ensure"] for row in bout_rows], dtype=bool),
    ensure_latency_s=np.asarray([row["ensure_latency_s"] for row in bout_rows], dtype=float),
)
print(f"Saved reusable all-bout cache: {CACHE_FILE.resolve()}", flush=True)
qc_rows

## Fast reanalysis from the saved cache

After the cache has been created, future sessions only need the import and configuration cells above, followed by this cell. Edit the filters and rerun this cell plus the figure/export cells; the `.doric` files are not reopened. `MIN_PRECEDING_LICK_SILENCE_S = 10.0` keeps only bouts with no detected lick during the preceding 10 seconds. Set it to `None` to disable that rule. Set `LICK_PLOT_MODE` to `"none"`, `"raster"`, `"rate"`, or `"both"`.

In [ ]:
ANALYSIS_MODE = "with_ensure"  # all, with_ensure, or without_ensure
MOUSE_FILTER = []  # Example: ["SC43", "SC44"]
DATE_FILTER = []  # Example: ["250625"]
RECORDING_TEXT = ""  # Case-insensitive substring, for example: "FoodReward"
MIN_LICKS_FILTER = None  # Example: 10; use None for no additional filter
MIN_PRECEDING_LICK_SILENCE_S = 5.0  # Use None to include bouts with recent licking
LICK_PLOT_MODE = "both"  # none, raster, rate, or both

if ANALYSIS_MODE not in {"all", "with_ensure", "without_ensure"}:
    raise ValueError(f"Unknown ANALYSIS_MODE: {ANALYSIS_MODE}")
if LICK_PLOT_MODE not in {"none", "raster", "rate", "both"}:
    raise ValueError(f"Unknown LICK_PLOT_MODE: {LICK_PLOT_MODE}")
if not CACHE_FILE.is_file():
    raise FileNotFoundError(f"Run the slow processing cell once to create {CACHE_FILE}")
with np.load(CACHE_FILE, allow_pickle=False) as cache:
    relative_time = cache["relative_time_s"]
    all_trials = cache["delta_tau_ns"]
    all_mice = cache["mouse"].astype(str)
    all_dates = cache["session_date"].astype(str)
    all_recordings = cache["recording"].astype(str)
    all_has_ensure = cache["has_ensure"].astype(bool)
    all_lick_counts = cache["lick_count"].astype(int)
    required_cache_fields = {"preceding_lick_gap_s", "lick_counts_per_bin", "lick_bin_width_s"}
    missing_cache_fields = required_cache_fields.difference(cache.files)
    if missing_cache_fields:
        raise ValueError(f"Cache is missing {sorted(missing_cache_fields)}; rerun the slow processing cell once")
    all_preceding_lick_gaps = cache["preceding_lick_gap_s"].astype(float)
    all_lick_count_trials = cache["lick_counts_per_bin"].astype(float)
    lick_bin_width_s = float(cache["lick_bin_width_s"])

selected = np.ones(all_trials.shape[0], dtype=bool)
if ANALYSIS_MODE == "with_ensure":
    selected &= all_has_ensure
elif ANALYSIS_MODE == "without_ensure":
    selected &= ~all_has_ensure
if MOUSE_FILTER:
    selected &= np.isin(all_mice, np.asarray(MOUSE_FILTER, dtype=str))
if DATE_FILTER:
    selected &= np.isin(all_dates, np.asarray(DATE_FILTER, dtype=str))
if RECORDING_TEXT:
    selected &= np.char.find(np.char.lower(all_recordings), RECORDING_TEXT.casefold()) >= 0
if MIN_LICKS_FILTER is not None:
    selected &= all_lick_counts >= int(MIN_LICKS_FILTER)
if MIN_PRECEDING_LICK_SILENCE_S is not None:
    selected &= all_preceding_lick_gaps >= float(MIN_PRECEDING_LICK_SILENCE_S)
if not np.any(selected):
    raise ValueError("No cached bouts match the current filters")

trials = all_trials[selected]
trial_mice = all_mice[selected]
trial_has_ensure = all_has_ensure[selected]
trial_dates = all_dates[selected]
trial_recordings = all_recordings[selected]
trial_lick_counts = all_lick_counts[selected]
trial_preceding_lick_gaps = all_preceding_lick_gaps[selected]
trial_lick_count_traces = all_lick_count_trials[selected]
print(f"Selected {trials.shape[0]} of {all_trials.shape[0]} cached bouts ({ANALYSIS_MODE})")
print(f"Minimum preceding lick-free interval: {MIN_PRECEDING_LICK_SILENCE_S}")
print(f"Mice: {', '.join(np.unique(trial_mice))}")
print(f"Recordings: {', '.join(np.unique(trial_recordings))}")

In [ ]:
mouse_names = np.unique(trial_mice)
mouse_means = np.vstack([np.nanmean(trials[trial_mice == mouse], axis=0) for mouse in mouse_names])
grand_mean = np.nanmean(mouse_means, axis=0)
valid_mice = np.sum(np.isfinite(mouse_means), axis=0)
grand_sem = (
    np.zeros_like(grand_mean)
    if mouse_means.shape[0] == 1
    else np.nanstd(mouse_means, axis=0, ddof=1) / np.sqrt(np.maximum(valid_mice, 1))
)
show_lick_raster = LICK_PLOT_MODE in {"raster", "both"}
show_lick_rate = LICK_PLOT_MODE in {"rate", "both"}
panel_count = 2 + int(show_lick_raster) + int(show_lick_rate)
height_ratios = [1.2, 1.0] + ([1.0] if show_lick_raster else []) + ([1.0] if show_lick_rate else [])
figure, axes = plt.subplots(
    panel_count,
    1,
    figsize=(11, 2.7 * panel_count),
    sharex=True,
    constrained_layout=True,
    gridspec_kw={"height_ratios": height_ratios},
)
axes = np.atleast_1d(axes)
figure.suptitle(f"Cached lick-bout analysis: {ANALYSIS_MODE.replace('_', ' ')}")
order = np.argsort(trial_has_ensure.astype(int))[::-1]
image = axes[0].imshow(
    trials[order],
    aspect="auto",
    origin="lower",
    extent=[relative_time[0], relative_time[-1], 0, trials.shape[0]],
    cmap="coolwarm",
    interpolation="nearest",
    resample=False,
)
axes[0].set_ylabel("Selected bout")
figure.colorbar(image, ax=axes[0], label="Delta tau (ns)")
for trace in mouse_means:
    axes[1].plot(relative_time, trace, color="0.75", linewidth=1)
axes[1].plot(relative_time, grand_mean, color="tab:blue", linewidth=2.5, label=f"Mouse mean (n={mouse_names.size})")
axes[1].fill_between(relative_time, grand_mean - grand_sem, grand_mean + grand_sem, color="tab:blue", alpha=0.2)
axes[1].set_ylabel("Delta tau (ns)")
axes[1].legend()
next_axis = 2
if show_lick_raster:
    axes[next_axis].imshow(
        (trial_lick_count_traces[order] > 0).astype(float),
        aspect="auto",
        origin="lower",
        extent=[relative_time[0], relative_time[-1], 0, trials.shape[0]],
        cmap="Greys",
        vmin=0,
        vmax=1,
        interpolation="nearest",
        resample=False,
    )
    axes[next_axis].set_ylabel("Lick raster")
    next_axis += 1
if show_lick_rate:
    lick_rate_trials = trial_lick_count_traces / lick_bin_width_s
    lick_mouse_means = np.vstack([np.nanmean(lick_rate_trials[trial_mice == mouse], axis=0) for mouse in mouse_names])
    lick_grand_mean = np.nanmean(lick_mouse_means, axis=0)
    lick_grand_sem = (
        np.zeros_like(lick_grand_mean)
        if lick_mouse_means.shape[0] == 1
        else np.nanstd(lick_mouse_means, axis=0, ddof=1) / np.sqrt(lick_mouse_means.shape[0])
    )
    for trace in lick_mouse_means:
        axes[next_axis].plot(relative_time, trace, color="0.75", linewidth=1)
    axes[next_axis].plot(relative_time, lick_grand_mean, color="tab:green", linewidth=2.5, label=f"Mouse mean (n={mouse_names.size})")
    axes[next_axis].fill_between(relative_time, lick_grand_mean - lick_grand_sem, lick_grand_mean + lick_grand_sem, color="tab:green", alpha=0.2)
    axes[next_axis].set_ylabel("Lick rate (Hz)")
    axes[next_axis].legend()
for axis in axes:
    axis.axvline(0, color="tab:orange", linestyle="--")
axes[-1].set_xlabel("Time from lick-bout onset (s)")
plt.show()

In [ ]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
def write_dict_rows(path, rows):
    if not rows:
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
if "qc_rows" in globals():
    write_dict_rows(OUTPUT_FOLDER / "recording_qc.csv", qc_rows)
if "bout_rows" in globals():
    write_dict_rows(OUTPUT_FOLDER / "bout_metadata.csv", bout_rows)
analysis_stem = f"bout_aligned_lifetime_{ANALYSIS_MODE}"
np.savez_compressed(
    OUTPUT_FOLDER / f"{analysis_stem}.npz",
    relative_time_s=relative_time,
    delta_tau_ns=trials,
    lick_counts_per_bin=trial_lick_count_traces,
    lick_bin_width_s=np.asarray(lick_bin_width_s),
    mouse=trial_mice,
    session_date=trial_dates,
    recording=trial_recordings,
    lick_count=trial_lick_counts,
    preceding_lick_gap_s=trial_preceding_lick_gaps,
    has_ensure=trial_has_ensure,
)
figure.savefig(OUTPUT_FOLDER / f"{analysis_stem}.png", dpi=200)
print(f"Wrote results to {OUTPUT_FOLDER.resolve()}")

Review `recording_qc.csv` before interpreting the aggregate. The slow stage writes `all_bout_cache.npz`; subsequent filter, figure, and export runs use only that cache. Use NI-DAQ alignment instead when licking, reward, cue, running, or other required events are not present in the `.doric` file.